# Check Enterprise RAG Benchmark

* Find the dataset here: https://github.com/onyx-dot-app/EnterpriseRAG-Bench
  * RAG leaderboard: https://huggingface.co/spaces/onyx-dot-app/EnterpriseRAG-Bench-Leaderboard
  * Known limitations: https://github.com/onyx-dot-app/EnterpriseRAG-Bench/blob/main/methodology.md#known-limitations
  * Future work: https://github.com/onyx-dot-app/EnterpriseRAG-Bench/blob/main/methodology.md#future-work
  * Repo layout: https://github.com/onyx-dot-app/EnterpriseRAG-Bench/blob/main/quickstart.md#repo-layout

* Observations 🍀
  * Their focus is literally company internal data, different from mine.
    * My Focus is using companies' internal data to serve for their external customers. --> customer facing RAGs.
  * Didn't find a way to link each Q&A to the original file

### Check Data

* `questions.jsonl`: The 500 benchmark questions
* `extra_questions.jsonl`: 100 additional metadata-dependent questions (not part of the leaderboard)
* Source files: https://github.com/onyx-dot-app/EnterpriseRAG-Bench/tree/main/generated_data/sources

In [6]:
from pathlib import Path
import json
import pandas as pd
from typing import List, Dict, Any
import os

def find_data_dir():
    candidates = [
        Path('enterpriseRAG_bench'),
        Path('datasets') / 'enterpriseRAG_bench',
        Path.cwd() / 'enterpriseRAG_bench',
        Path.cwd() / 'datasets' / 'enterpriseRAG_bench',
    ]
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    # fallback: search recursively for folder name (case-insensitive)
    for d in Path.cwd().rglob('*'):
        if d.is_dir() and d.name.lower() == 'enterpriserag_bench':
            return d
        if d.is_dir() and d.name.lower().startswith('enterprise') and 'rag' in d.name.lower():
            return d
    raise FileNotFoundError(
        'Could not locate enterpriseRAG_bench directory. Checked: ' + ','.join(str(x) for x in candidates)
    )

def load_file(path: Path) -> Dict[str, Any]:
    suffix = path.suffix.lower()
    if suffix in ('.jsonl', '.ndjson'):
        items = []
        with path.open('r', encoding='utf8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                items.append(json.loads(line))
        return {'path': str(path), 'type': 'jsonl', 'content': items}
    if suffix == '.json':
        with path.open('r', encoding='utf8') as f:
            return {'path': str(path), 'type': 'json', 'content': json.load(f)}
    if suffix in ('.csv', '.tsv'):
        sep = '\t' if suffix == '.tsv' else ','
        df = pd.read_csv(path, sep=sep)
        return {'path': str(path), 'type': 'table', 'content': df.to_dict(orient='records')}
    if suffix in ('.txt', '.md'):
        return {'path': str(path), 'type': 'text', 'content': path.read_text(encoding='utf8')}
    return {'path': str(path), 'type': 'binary', 'content': None}

def load_enterprise_rag_bench(data_dir: Path = None) -> List[Dict[str, Any]]:
    if data_dir is None:
        data_dir = find_data_dir()
    out = []
    for p in sorted(data_dir.rglob('*')):
        if p.is_file():
            try:
                out.append(load_file(p))
            except Exception as e:
                out.append({'path': str(p), 'type': 'error', 'error': str(e)})
    return out


In [7]:
# Load dataset and print examples
import pprint

try:
    data_dir = find_data_dir()
    print('Found data dir:', data_dir)
    structured = load_enterprise_rag_bench(data_dir)
    print(f'Loaded {len(structured)} files\n' + '='*60)

    for item in structured[:5]:
        print(f"\nFile : {item['path']}")
        print(f"Type : {item['type']}")
        content = item.get('content')
        if isinstance(content, list):
            print(f"Count: {len(content)} records")
            print("Sample (first 2 records):")
            pprint.pprint(content[:2], indent=2, width=100)
        elif isinstance(content, str):
            print("Sample (first 500 chars):")
            print(content[:500])
        elif content is not None:
            pprint.pprint(content, indent=2, width=100)
        print('-' * 60)
except FileNotFoundError as e:
    print(e)

Found data dir: enterpriseRAG_bench
Loaded 2 files

File : enterpriseRAG_bench/extra_questions.jsonl
Type : jsonl
Count: 100 records
Sample (first 2 records):
[ { 'answer_facts': [ 'The draft modeling spreadsheet about continuous headcount ramping and small '
                      'price changes to assess runway sensitivity is owned by Maya Chen.'],
    'expected_doc_ids': ['dsid_fc36d1d60e7e4b4abc7db84629563b7a'],
    'gold_answer': 'The draft modeling spreadsheet is owned by Maya Chen.',
    'question': 'In the finance operations shared drive, who owns the draft modeling spreadsheet '
                'about continuous headcount ramping and small price changes to assess runway '
                'sensitivity?',
    'question_id': 'qst_0001',
    'question_type': 'metadata',
    'source_types': ['google_drive']},
  { 'answer_facts': [ 'The internal organizer for the February 2025 security review call was '
                      'Markus Klein (Redwood).'],
    'expected_doc_ids': ['dsid_